In [ ]:
import os
import zipfile
import shutil
import uuid
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.utils.class_weight import compute_class_weight

from google.colab import drive
drive.mount('/content/drive')

# ================================
# CONFIGURACIÓN GENERAL
# ================================
IMG_SIZE   = (256, 320)
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 0.001
SEED       = 123
THRESHOLD  = 0.40    # umbral de decisión (score >= THRESHOLD → MENOR)

MODEL_DIR = '/content/modelos_imagenes_menores'
os.makedirs(MODEL_DIR, exist_ok=True)

ZIP_PATH     = '/content/drive/MyDrive/imagenes_menores.zip'
EXTRACT_PATH = '/content/dataset'
SRC_PATH     = '/content/dataset/face_age'
DST_PATH     = '/content/dataset_clasificado'

print('Setup completado')

In [ ]:
# ================================
# FUNCIONES DE PREPARACIÓN
# ================================

def descomprimir_dataset(zip_path, extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print('Dataset descomprimido')

def encontrar_face_age(base_path):
    for root, dirs, files in os.walk(base_path):
        if 'face_age' in dirs:
            return os.path.join(root, 'face_age')
    return None

def clasificar_dataset(src_path, dst_path):
    os.makedirs(dst_path + '/menor', exist_ok=True)
    os.makedirs(dst_path + '/mayor', exist_ok=True)
    for folder in os.listdir(src_path):
        folder_path = os.path.join(src_path, folder)
        if not folder.isdigit():
            continue
        edad = int(folder)
        for img in os.listdir(folder_path):
            src_img = os.path.join(folder_path, img)
            if not os.path.isfile(src_img):
                continue
            nombre = f'{edad}_{uuid.uuid4().hex[:8]}.jpg'
            destino = 'menor' if edad < 18 else 'mayor'
            shutil.copy(src_img, os.path.join(dst_path, destino, nombre))
    print('Dataset clasificado')

def ver_distribucion(dst_path):
    menores = len(os.listdir(dst_path + '/menor'))
    mayores = len(os.listdir(dst_path + '/mayor'))
    print(f'Menores: {menores} | Mayores: {mayores}')

In [ ]:
# ================================
# PREPARAR DATASET
# ================================

if not os.path.exists(DST_PATH):
    print('Dataset no encontrado, procesando...')
    descomprimir_dataset(ZIP_PATH, EXTRACT_PATH)
    SRC_PATH = encontrar_face_age(EXTRACT_PATH)
    if SRC_PATH is None:
        raise ValueError('No se encontró la carpeta face_age')
    clasificar_dataset(SRC_PATH, DST_PATH)
    ver_distribucion(DST_PATH)
else:
    print('Dataset ya preparado')
    ver_distribucion(DST_PATH)

def get_imagenes(dst_path=DST_PATH):
    train_dataset = image_dataset_from_directory(
        dst_path,
        validation_split=0.2,
        subset='training',
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='int'
    )
    validation_dataset = image_dataset_from_directory(
        dst_path,
        validation_split=0.2,
        subset='validation',
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='int'
    )
    print('Clases detectadas:', train_dataset.class_names)
    return train_dataset, validation_dataset

train_d, validation_d = get_imagenes()
class_names = train_d.class_names
print(f'label 0 = {class_names[0]} | label 1 = {class_names[1]}')
print(f'score >= {THRESHOLD}  →  MENOR')

# Balanceo de clases
n_menores = len(os.listdir(DST_PATH + '/menor'))
n_mayores = len(os.listdir(DST_PATH + '/mayor'))
labels_array = np.array([0] * n_mayores + [1] * n_menores)
weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=labels_array)
class_weight = dict(enumerate(weights))
print(f'class_weight: {class_weight}')

In [ ]:
# ================================
# VISUALIZAR MUESTRAS
# ================================

plt.figure(figsize=(10, 10))
for images, labels in train_d.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')

In [ ]:
# ================================
# ARQUITECTURA CNN BÁSICA
# ================================

def get_model_basico(
    input_shape=(256, 320, 3),
    conv1=32, conv2=64, conv3=128,
    dense1=64, dense2=32,
    kernel1=(3, 3), kernel2=(5, 5), kernel3=(3, 3),
    stride2=2
):
    inp = layers.Input(shape=input_shape)

    # Bloque 1
    x = layers.Conv2D(conv1, kernel1, padding='same', activation='relu')(inp)
    x = layers.Conv2D(conv1, kernel1, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    # Bloque 2
    x = layers.Conv2D(conv2, kernel2, strides=stride2, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    # Bloque 3
    x = layers.Conv2D(conv3, kernel3, padding='same', activation='relu')(x)

    # Clasificador
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense1, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(dense2, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=out)
    model.summary()
    return model

In [ ]:
# ================================
# NORMALIZAR
# ================================

def normalizar(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.cast(label, tf.float32)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_norm = train_d.map(normalizar).prefetch(AUTOTUNE)
val_norm   = validation_d.map(normalizar).prefetch(AUTOTUNE)

In [ ]:
# ================================
# CALLBACKS Y FUNCIÓN DE GRÁFICAS
# ================================

def graficar(history, title=''):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='train')
    plt.plot(history.history['val_accuracy'], label='val')
    plt.title(f'{title} — Accuracy')
    plt.ylim(0, 1)
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.title(f'{title} — Loss')
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

def get_callbacks(model_name):
    path = os.path.join(MODEL_DIR, f'{model_name}.keras')
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=path,
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    ]

## Experimento 1 — CNN básica sin augmentation

In [ ]:
modelo_cnn = get_model_basico()

modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_cnn = modelo_cnn.fit(
    train_norm,
    validation_data=val_norm,
    epochs=EPOCHS,
    callbacks=get_callbacks('cnn_sin_aug'),
    class_weight=class_weight
)

In [ ]:
graficar(history_cnn, 'Modelo sin aumento de datos')

## Experimento 2 — CNN básica con augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def augmentar(image, label):
    image = data_augmentation(image, training=True)
    return image, label

train_aug = train_norm.map(augmentar).prefetch(AUTOTUNE)
val_aug   = val_norm   # validación sin augmentation

In [ ]:
# Reutilizamos el mismo modelo (continúa desde los pesos del exp. 1)
modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_aug = modelo_cnn.fit(
    train_aug,
    validation_data=val_aug,
    epochs=EPOCHS,
    callbacks=get_callbacks('cnn_con_aug'),
    class_weight=class_weight
)

In [ ]:
graficar(history_aug, 'Modelo con aumento de datos')

## Experimento 3 — ResNet50 (CORREGIDO)

**Problema en la versión original:**
1. Se aplicaba `/255` al dataset y luego ResNet50 recibía imágenes ya en `[0, 1]`. ResNet50 espera su propio `preprocess_input` (sustracción de medias ImageNet por canal), por lo que la entrada era incorrecta y el modelo se quedaba estancado en ~60%.
2. El `ModelCheckpoint` heredaba el baseline de 0.857 del CNN anterior, por lo que ResNet nunca llegaba a guardar aunque mejorara.

**Fix:**
- Dataset nuevo sin `/255`, con `resnet_preprocess` aplicado en el pipeline.
- Callbacks propios con path y baseline independientes.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# Dataset sin /255 — ResNet50 hace su propio preprocesado
train_res_raw, val_res_raw = get_imagenes()

def preprocess_resnet(image, label):
    image = tf.cast(image, tf.float32)
    image = resnet_preprocess(image)   # sustracción de medias ImageNet
    label = tf.cast(label, tf.float32)
    return image, label

train_resnet = train_res_raw.map(preprocess_resnet).map(augmentar).prefetch(AUTOTUNE)
val_resnet   = val_res_raw.map(preprocess_resnet).prefetch(AUTOTUNE)

In [ ]:
modelo_resnet = get_resnet()

modelo_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_resnet = modelo_resnet.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=EPOCHS,
    callbacks=get_callbacks('resnet_fase1'),
    class_weight=class_weight
)

In [ ]:
graficar(history_resnet, 'ResNet50 — Fase 1')

## Fine-tuning ResNet50 (opcional)

Descongelamos los últimos 30 layers de ResNet50 con un learning rate mucho más bajo para ajustar los pesos al dominio de caras.

In [ ]:
base = modelo_resnet.layers[1]   # ResNet50
base.trainable = True

fine_tune_from = len(base.layers) - 30
for layer in base.layers[:fine_tune_from]:
    layer.trainable = False

print(f'Capas entrenables en ResNet50: {sum(1 for l in base.layers if l.trainable)} de {len(base.layers)}')

modelo_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(LR / 10),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_finetune = modelo_resnet.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=15,
    callbacks=get_callbacks('resnet_fase2'),
    class_weight=class_weight
)

In [ ]:
graficar(history_finetune, 'ResNet50 — Fine-tuning')

## Comparación final y guardado del mejor modelo

In [ ]:
def evaluar(model, dataset, threshold=THRESHOLD):
    y_true, y_pred = [], []
    for imgs, labels in dataset:
        scores = model.predict(imgs, verbose=0).flatten()
        y_pred.extend((scores >= threshold).astype(int))
        y_true.extend(labels.numpy().astype(int))
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = np.mean(y_true == y_pred)
    tp  = np.sum((y_pred == 1) & (y_true == 1))
    fp  = np.sum((y_pred == 1) & (y_true == 0))
    fn  = np.sum((y_pred == 0) & (y_true == 1))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    return acc, precision, recall

acc_cnn,    prec_cnn,    rec_cnn    = evaluar(modelo_cnn,    val_aug)
acc_resnet, prec_resnet, rec_resnet = evaluar(modelo_resnet, val_resnet)

print(f'Threshold aplicado: {THRESHOLD}  (score >= {THRESHOLD} → MENOR)\n')
print(f'{"Modelo":<22} {"Accuracy":>10} {"Precision":>10} {"Recall":>10}')
print('-' * 54)
print(f'{"CNN básica + aug":<22} {acc_cnn:>10.4f} {prec_cnn:>10.4f} {rec_cnn:>10.4f}')
print(f'{"ResNet50":<22} {acc_resnet:>10.4f} {prec_resnet:>10.4f} {rec_resnet:>10.4f}')

mejor        = modelo_resnet if acc_resnet >= acc_cnn else modelo_cnn
nombre_mejor = 'ResNet50' if acc_resnet >= acc_cnn else 'CNN básica'

OUTPUT_PATH = os.path.join(MODEL_DIR, 'modelo_menores.h5')
mejor.save(OUTPUT_PATH)

print(f'\nMejor modelo: {nombre_mejor}')
print(f'Guardado en:  {OUTPUT_PATH}')